# Clase 5 — Sesgos y justicia algorítmica

> *"Los algoritmos no son neutrales: heredan los sesgos de los datos con los que se entrenan
> y de las personas que los diseñan."*

En esta clase vas a entender **por qué un modelo aparentemente "objetivo" puede discriminar**,
ver **ejemplos cotidianos** de sesgo algorítmico, y conocer estrategias para
detectar y mitigar sesgos.

### Lo que vas a llevarte

| Sección | Idea |
|---|---|
| 1 | ¿Por qué un modelo puede ser injusto? |
| 2 | 4 ejemplos cotidianos de sesgo algorítmico |
| 3 | De dónde vienen los sesgos |
| 4 | Demo: un dataset sesgado en acción |
| 5 | Cómo se mide la "justicia" de un modelo |
| 6 | Estrategias de mitigación |
| 7 | Ejercicio: auditá un caso |

_
> ⚖️ Esta clase conecta con el **Módulo 1 → Ética y justicia algorítmica**
> del programa oficial. Es contenido **evaluable** del TP 1.


---
## 1. ¿Por qué un modelo puede ser injusto?

Los modelos de ML aprenden **patrones estadísticos** de los datos.
Si los datos **reflejan desigualdades históricas**, el modelo las **reproduce y amplifica**.

### La falacia de la "neutralidad matemática"

> *"Pero el modelo no sabe el género/raza/edad de la persona..."*

**No importa.** El modelo descubre **proxies**: variables aparentemente neutras que
**correlacionan** con atributos sensibles.

| Variable "neutra" | Proxy que esconde |
|---|---|
| Código postal | Nivel socioeconómico, raza |
| Nombre de pila | Género, origen étnico |
| Historial de empleo con pausas | Maternidad |
| Tipo de escuela secundaria | Clase social |
| Patrón de uso del teléfono | Edad, discapacidad |

> ⚠️ Quitar la variable sensible del dataset **no elimina el sesgo**.
> Solo lo hace **invisible y más difícil de auditar**.


---
## 2. Cuatro ejemplos cotidianos de sesgo algorítmico

### 💳 Scoring crediticio para trabajadores independientes

Modelo para estimar el **riesgo de mora** al pedir un préstamo.

- **Qué pasó:** el sistema rechazaba más a monotributistas, freelancers o personas con ingresos variables, incluso con buen historial de pago.
- **Por qué:** aprendía que tener un ingreso fijo y predecible era un proxy demasiado fuerte de bajo riesgo.
- **Impacto:** exclusión financiera de perfiles solventes, pero distintos del cliente histórico "ideal".

### 🛒 Detección de fraude en compras online

Sistema que bloquea transacciones **sospechosas** con tarjeta o billetera digital.

- **Qué pasó:** compras legítimas durante viajes, mudanzas o eventos excepcionales se marcaban como fraude con mucha más frecuencia.
- **Por qué:** el modelo estaba entrenado sobre patrones "normales" y castigaba comportamientos poco frecuentes aunque fueran válidos.
- **Impacto:** usuarios bloqueados justo cuando más necesitaban operar.

### 🎓 Asignación automática de becas o admisiones

Modelo que **rankea postulantes** para becas, cupos o programas.

- **Qué pasó:** personas de escuelas menos conocidas o con trayectorias no tradicionales quedaban sistemáticamente peor posicionadas.
- **Por qué:** variables como escuela de origen, actividades extraescolares o nivel de inglés funcionaban como proxies de contexto socioeconómico.
- **Impacto:** el sistema reforzaba desigualdades previas en lugar de corregirlas.

### 🚚 Predicción de tiempos de entrega o atención

Modelo que estima **demoras** en logística, soporte o turnos.

- **Qué pasó:** zonas rurales, rutas poco frecuentes o picos de demanda tenían errores mucho mayores que el promedio.
- **Por qué:** había menos datos de entrenamiento de esos escenarios, así que el modelo funcionaba bien en lo común y mal en lo excepcional.
- **Impacto:** peor servicio y decisiones operativas inconsistentes entre zonas o tipos de clientes.

> 📚 El sesgo no siempre aparece como un gran escándalo público: muchas veces entra por
> comodidad operativa, datos incompletos o proxies mal elegidos.


---
## 3. ¿De dónde vienen los sesgos?

No hay una sola causa. Hay **toda una cadena**:

| Etapa | Tipo de sesgo | Ejemplo |
|---|---|---|
| **Recolección** | Muestreo sesgado | Encuesta online → excluye personas sin internet |
| **Etiquetado** | Sesgo del anotador | Personas de un país etiquetan emociones según su cultura |
| **Variables elegidas** | Proxies ocultos | Usar "código postal" como feature |
| **Definición del problema** | Objetivo mal planteado | "Predecir éxito" = "predecir a quién contratamos antes" |
| **Distribución histórica** | Sesgo histórico | Datos de arrestos con sesgo policial |
| **Feedback loop** | Amplificación en producción | Modelo recomienda → usuarios consumen → confirma modelo |

### El feedback loop es el más peligroso

```
  Modelo predice    →    Decisión     →    Se genera   →    Nuevos datos
  "esta zona es         "mandar más        más arrestos     "la zona SÍ
   peligrosa"            policía allá"      en esa zona      era peligrosa"
       ▲                                                            │
       └────────────────────────────────────────────────────────────┘
                          el sesgo se AUTOREFUERZA
```


---
## 4. Demo: un dataset sesgado en acción

Vamos a simular un caso típico: un modelo que decide si aprobar un **crédito**,
entrenado con datos donde históricamente se aprobaron **más hombres que mujeres**.


In [3]:
# Simulación mínima: generar dataset sesgado y ver el efecto
import random
random.seed(42)

# Generamos 1000 solicitantes ficticios con capacidad de pago similar entre géneros
solicitantes = []
for _ in range(1000):
    genero = random.choice(["M", "F"])
    ingreso = random.gauss(50, 15)  # mismo ingreso promedio para ambos
    deuda   = random.gauss(10, 5)
    # La capacidad real de pago NO depende del género
    paga_real = (ingreso - deuda) > 30

    # Pero la decisión HISTÓRICA sí estaba sesgada: se aprobaba más a hombres
    # aunque la capacidad fuera igual
    if genero == "M":
        aprobado_hist = paga_real or (random.random() < 0.2)  # se les dio el beneficio de la duda
    else:
        aprobado_hist = paga_real and (random.random() < 0.7)  # se les exigió más

    solicitantes.append({
        "genero": genero, "ingreso": ingreso, "deuda": deuda,
        "paga_real": paga_real, "aprobado_historico": aprobado_hist
    })

# Resumen
def tasa(key, filtro):
    subset = [s for s in solicitantes if filtro(s)]
    return sum(s[key] for s in subset) / len(subset) if subset else 0

print("Dataset histórico (lo que el modelo aprendería):\n")
print(f"  Tasa real de pago — Hombres:  {tasa('paga_real',        lambda s: s['genero']=='M'):.1%}")
print(f"  Tasa real de pago — Mujeres:  {tasa('paga_real',        lambda s: s['genero']=='F'):.1%}")
print(f"  Tasa de aprobación histórica — Hombres: {tasa('aprobado_historico', lambda s: s['genero']=='M'):.1%}")
print(f"  Tasa de aprobación histórica — Mujeres: {tasa('aprobado_historico', lambda s: s['genero']=='F'):.1%}")
print("\n⚠️ Capacidad de pago parecida, aprobaciones MUY distintas.")
print("   Un modelo entrenado con esto aprendería 'mujer = más riesgo', aunque no sea cierto.")


Dataset histórico (lo que el modelo aprendería):

  Tasa real de pago — Hombres:  72.1%
  Tasa real de pago — Mujeres:  71.5%
  Tasa de aprobación histórica — Hombres: 76.9%
  Tasa de aprobación histórica — Mujeres: 49.8%

⚠️ Capacidad de pago parecida, aprobaciones MUY distintas.
   Un modelo entrenado con esto aprendería 'mujer = más riesgo', aunque no sea cierto.


> 🔍 **¿Qué pasó?** Aunque la capacidad **real** de pago es similar entre géneros,
> la etiqueta del dataset (`aprobado_historico`) está sesgada. Un modelo entrenado
> con esta etiqueta como target aprendería a **rechazar más mujeres**, incluso si
> nunca le pasamos el género como feature (lo inferiría por proxies).
>
> Este es **exactamente** el mecanismo que aparece cuando la etiqueta histórica
> está sesgada: el modelo aprende a repetir decisiones viejas, no a estimar mejor
> la realidad.


---
## 5. ¿Cómo se mide la "justicia" de un modelo?

Hay **muchas métricas de fairness**, y suelen ser **incompatibles entre sí**.
Acá vamos a ver las 3 más intuitivas.

### Antes de mirar las fórmulas: cómo se leen

Si nunca viste esta notación, usá esta guía rápida:

- **$P( ... )$** significa **"probabilidad de que pase algo"**.
- **$\hat{y}=1$** significa **"el modelo predice positivo"**.
- **$Y=1$** significa **"en la realidad el caso era positivo"**.
- **$A$** representa el **grupo** que queremos comparar.
- **$A=a$** y **$A=b$** representan **dos grupos distintos**.
- **$\mid$** se lee **"dado que"**.

En un ejemplo de crédito:

- **$\hat{y}=1$** puede significar **"el modelo aprueba el crédito"**.
- **$Y=1$** puede significar **"la persona realmente iba a pagar"**.

> 📌 Idea clave: casi todas estas fórmulas comparan **tasas** entre grupos.

### Paridad demográfica

> *"La tasa de aprobación debe ser igual entre grupos."*

$$P(\hat{y}=1 \mid A=a) = P(\hat{y}=1 \mid A=b)$$

**Se lee así:** la probabilidad de que el modelo dé un resultado positivo,
**dado que** la persona pertenece al grupo A, debe ser igual a la del grupo B.

**En simple:** si el modelo aprueba al 40% de un grupo, no debería aprobar al 70%
de otro grupo sin una razón muy clara y justificada.

- ✅ Simple y auditable.
- ❌ Puede forzar aprobar a quien no califica.

### Igualdad de oportunidad

> *"Entre quienes SÍ merecen el positivo, la tasa debe ser igual."*

$$P(\hat{y}=1 \mid Y=1, A=a) = P(\hat{y}=1 \mid Y=1, A=b)$$

**Se lee así:** entre los casos que en la realidad eran positivos,
la probabilidad de que el modelo acierte debe ser la misma en ambos grupos.

**En simple:** si en dos grupos hay personas que sí iban a pagar un crédito,
el modelo no debería dejar afuera más buenos casos de un grupo que del otro.

- ✅ Respeta el mérito.
- ❌ Requiere conocer la "verdad" (`Y`), que muchas veces ya está sesgada.

### Calibración

> *"Si el modelo dice '80% de probabilidad', debe acertar 80% de las veces en CADA grupo."*

$$P(Y=1 \mid \hat{p}=0.8, A=a) = 0.8$$

**Se lee así:** si el modelo asigna 0.8 de probabilidad a un caso del grupo A,
entonces alrededor del 80% de esos casos debería ser realmente positivo.

**En simple:** cuando el modelo dice "muy probable", ese porcentaje debería significar
lo mismo para todos los grupos.

- ✅ Útil en scoring (crédito, riesgo médico).
- ❌ Incompatible con igualdad de oportunidad en la mayoría de casos reales
  (resultado de Kleinberg et al., 2016).


_
**Resumen rápido:**

- **Paridad demográfica** mira si el modelo aprueba en proporciones parecidas.
- **Igualdad de oportunidad** mira si el modelo acierta parecido entre quienes sí merecían el positivo.
- **Calibración** mira si los porcentajes del modelo significan lo mismo para todos.


_
> ⚖️ **No existe una definición única de "justo"**. Elegir la métrica es una **decisión política**,
> no técnica. Debe documentarse en la ficha del proyecto (ISO 42001, clase 7).


---
## 6. Estrategias de mitigación

Las intervenciones se aplican en distintos puntos del pipeline:

### Pre-procesamiento (sobre los datos)

- **Re-muestreo:** balancear grupos subrepresentados.
- **Re-etiquetado:** corregir etiquetas históricamente sesgadas.
- **Supresión cuidadosa de proxies** (no solo variables sensibles, también las correlacionadas).

### In-procesamiento (durante el entrenamiento)

- **Restricciones de fairness** como parte de la función de pérdida.
- **Adversarial debiasing:** un segundo modelo castiga al principal si predice el atributo sensible.

### Post-procesamiento (sobre las predicciones)

- **Umbrales distintos por grupo** para igualar tasas.
- **Calibración por grupo.**

### Humanos en el loop

| Elemento | Por qué importa |
|---|---|
| 📋 **Model cards** | Documentar datos, limitaciones, grupos evaluados |
| 🔍 **Auditorías externas** | Independientes del equipo que construye el modelo |
| 🚨 **Canales de reporte** | Que los afectados puedan denunciar decisiones |
| ⛔ **Derecho a no-automatización** | Revisión humana de decisiones críticas (exigido por el AI Act) |

_
> 🛠️ Mitigar sesgos **no es una sola acción técnica**: es una práctica continua que
> combina ingeniería, gobernanza y participación de los afectados.


# Preparación de Entorno de Trabajo y Ejercicios

Requisitos:
- Descargar Visual Studio Code (VSCode): https://code.visualstudio.com/download
- Instalar la extensión **Python**
- Instalar la extensión para manejar los entornos **Python Environments**
- Crear nuestro primer archivo Markdown (extensión `.md`) para responder las preguntas.

**Ejercicio**:

Elegí **uno** de estos sistemas (o proponé otro):

- Un sistema que **filtra CVs** automáticamente en una plataforma de empleo.
- Un algoritmo que decide a quién le cobran **más caro el seguro del auto**.
- Un modelo que recomienda **a quién darle beca** en una universidad.
- Un sistema que **asigna turnos médicos** según prioridad estimada.

Respondé en un documento de **máximo 1 página**:

1. ¿Qué **datos históricos** probablemente alimenten al modelo?
2. ¿Qué **proxies** sensibles podrían aparecer?
3. ¿Qué **grupo** podría salir perjudicado?
4. ¿Qué **métrica de fairness** elegirías para auditarlo y por qué?
5. Proponé **2 mitigaciones concretas**.

---
### **Tips de uso de markdown en VSCode:**

# `#` para H1, 
## `##` para H2, 
### `###` para H3, 
###### hasta `######` para H6.

==========================================

Para textos:
- **negrita**, usá `**texto**`.
- *cursiva*, usá `*texto*`.



Para listas, 
- `-` o `*` para bullets,
    * si usás `-` o `*` también se pueden anidar con sangría.
1. `1.` numeros para listas ordenadas.

==========================================


Para tablas, usá la sintaxis de markdown:

```
| Columna 1 | Columna 2 |
|---|---|
| Dato 1 | Dato 2 |
```
Da como resultado:
| Columna 1 | Columna 2 |
|---|---|
| Dato 1 | Dato 2 |